In [ ]:
import sys
PATH = "your/path/to/PRIVET/repo"
sys.path.append(f"{PATH}/PRIVET")

import time

from src.misc_utils import *
from src.data_utils import *
from src.nn_utils import *
from src.stats_utils import *
from src.plot_utils import *
from src.privet import *

In [ ]:
ngpu=1
device = torch.device("cuda:0" if (torch.cuda.is_available() and ngpu > 0) else "cpu")
print(device)

In [ ]:
PATH = "your/path/to/data"
train = np.load(f"{PATH}/HGD_TRAIN_70-30_SPLIT.npy",allow_pickle=True)
test = np.load(f"{PATH}/HGD_TEST_70-30_SPLIT.npy",allow_pickle=True)

print(train.shape, test.shape)

Xtrain = np.vstack((train, test))
Xtrain = torch.tensor(Xtrain)

In [ ]:
# the EVT fit will be done on the fusion of train and test (also called val in paper)
# the goal is to retrieve train among concat(train,test)

############################
## COMPUTE 1-NN distances ##
############################
#Train-Train
dist_NN_tr_tr,  dist_NN_tr_tr_idx = gpu_nearest_neighbors(Xtrain, k=1,distance='hamming',chunk_size=256,device=device.type,verbose=False)
p_tr_tr_NN_dist, p_tr_tr_NN_idx = sorting(dist_NN_tr_tr, dist_NN_tr_tr_idx)
p_tr_tr_NN_dist, p_tr_tr_NN_idx = np.array(p_tr_tr_NN_dist), np.array(p_tr_tr_NN_idx)

In [ ]:
##############################################
## EVT FIT CDF on (Train+Test)-(Train+Test) ##
##############################################

N_train = Xtrain.shape[0]
partition_start = 0.001 #this is a hyperparameter of PRIVET
partition_end = 0.05 #this is a hyperparameter of PRIVET
#default would be to fit on 1% to 20%. but depending on shape of CDF (if there are multiple modes), 
#we want to fit the first mode, because we are interested in lower tail
#to be 100% sure on the partition of the fit, visual inspection of the CDF that you want to fit is recommended

start = int(np.ceil(partition_start*N_train)) #int(0.01*N) # if N is small start = 0 --> problem with log
end = int(partition_end*N_train)

# Fit parameters (adjust start/end indices to avoid extremes)
intercept, alpha, std_err_intercept, std_err_alpha, sigma_Y_pred = fit_nearest_neighbor_cdf_weibull(p_tr_tr_NN_dist.reshape(-1,), start_idx=start, end_idx=end)

print(f"Estimated intercept = {intercept:.2f} ± {std_err_intercept:.2f}")
print(f"Estimated alpha = {alpha:.2f} ± {std_err_alpha:.2f}")

In [ ]:
store_scores_4plot = []

names = [1,3,7,12, -1]
for i, name in enumerate(names):
    print(name)
    if name > 0:
        synth = np.load(f"{PATH}/PTT_Exp4_70-30_HGD_{name}.npy",allow_pickle=True)
    else:
        synth = np.load(f"{PATH}/DNA/Exp4_100k.npy",allow_pickle=True)
    synth = torch.tensor(synth)
    
    _lambda = synth.shape[0]/Xtrain.shape[0]

    store, p_tr = PRIVET_inverse(
        Xtrain, synth,
        intercept, alpha,
        renormalization=_lambda**(-1/55),
        distance='hamming',
        device=device.type
    )

    store_scores_4plot.append(store)

In [ ]:
FONTSIZE=13

plt.rcParams.update({
    'axes.labelsize': FONTSIZE,
    'axes.titlesize': FONTSIZE,
    'xtick.labelsize': FONTSIZE,
    'ytick.labelsize': FONTSIZE
})

fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(4,4), dpi=200, facecolor="white")

cmap = plt.cm.viridis_r
colors = cmap(np.linspace(0, 1, len(names)))  # generates n RGBA colors evenly spaced

for i, name in enumerate(names):
    store = store_scores_4plot[i]

    flist = np.zeros((Xtrain.shape[0],))
    flist[np.where(store[:,-2].astype(int)<3500)[0]] = 1
    flist=flist.astype(bool)

    scores = np.log10(store[:,0])
    
    unique_vals = np.sort(np.unique(scores))
    # Compute midpoints between consecutive unique values
    midpoints = (unique_vals[:-1] + unique_vals[1:]) / 2
    # Optionally, add thresholds at the boundaries to cover extremes:
    thresholds = np.concatenate(([unique_vals[0] - 1], midpoints, [unique_vals[-1] + 1]))
    
    x_fpr, y_tpr, auc_roc, y_precision, x_recall, auc_pr, threshold_lst, tp_lst, fp_lst, tn_lst, fn_lst = roc(flist, scores, thresholds)

    if name>0:
        label_name = str(name)+" (10K synth)"
    else:
        label_name = str(12)+" (100K synth)"
    ax.plot(x_recall,y_precision,label=f"RBM{label_name}, AUC={np.round(auc_pr,2)}",marker=".",markersize=1,linewidth=1,color=colors[i],alpha=.5)

ax.set_xlim(-0.05, 1.05)
ax.set_ylim(-0.05, 1.05)
ax.set_xticks([0, 0.2, 0.4, 0.6, 0.8, 1.0])
ax.set_xlabel(r"Recall ($\frac{tp}{tp + fn}$)")
ax.set_ylabel(r"Precision ($\frac{tp}{tp + fp}$)")
ax.grid(True)

leg = ax.legend(prop={'size': 9})
for lh in leg.legend_handles:
    lh.set_alpha(1)

fig.tight_layout()
plt.show()
